# Part 3.1 The deployed agent

After this notebook you can call the rebooking agent as a service, run a conversation on it, and
approve the credit it asks about.

In [ ]:
from kea.config import setup

setup("kea-rebooking-local")

## See it

The `rebooking` agent from Part 2 is running as a deployment named `kea-rebooking`. Nothing about
the agent changed. It now sits behind a URL, and every run lands in a tracing project with online
evaluator scores next to it.

<img src="../../assets/slides/adlc-5-on-langsmith.jpg" width="720">

In the lifecycle, this part covers **Deploy** (Agent Server) and **Monitor** (tracing, online evals).

<img src="../../assets/screenshots/p3-01-deployment.png" alt="LangSmith → Deployments → kea-rebooking: status Ready, the deployment URL, and the revision list." width="900">

<img src="../../assets/screenshots/p3-01-online-scores.png" alt="the deployment's tracing project, Traces tab: recent runs with the piileakage, toolselection, and creditwithoutapproval feedback columns filled in." width="900">

To reproduce: open LangSmith, choose **Deployments**, open `kea-rebooking`, then follow the link to
its tracing project.

## Build it

Put the deployment URL in `.env` as `KEA_DEPLOYMENT_URL`. Then connect with the LangGraph SDK and
list what the server hosts.

In [ ]:
# needs: deployment
import os

from langgraph_sdk import get_client

DEPLOYMENT_URL = os.environ.get("KEA_DEPLOYMENT_URL")
client = None
if DEPLOYMENT_URL:
    client = get_client(url=DEPLOYMENT_URL, api_key=os.environ["LANGSMITH_API_KEY"])
else:
    print(
        "KEA_DEPLOYMENT_URL is not set, so the deployment cells are skipped. Add the deployment URL to .env, restart the kernel, and run again."
    )

if client:
    for assistant in await client.assistants.search():
        print(assistant["graph_id"], assistant["assistant_id"])

Run one conversation. A **thread** holds the conversation, and each message you send is a **run**
on it. The passenger asks for a credit. The agent's `issue_credit` tool needs a human to approve
it, so the run stops at an interrupt.

In [ ]:
# needs: deployment
def reply(result: dict) -> str:
    last = result["messages"][-1]
    content = last["content"] if isinstance(last, dict) else last.content
    if isinstance(content, list):
        content = " ".join(block.get("text", "") for block in content)
    return content


thread, pending = None, None
if client:
    thread = await client.threads.create()
    result = await client.runs.wait(
        thread["thread_id"],
        "rebooking",
        input={
            "messages": [
                {
                    "role": "user",
                    "content": "KA 1182 DAL to DEN was cancelled. Confirmation ABC123, last name "
                    "Example. Rebook me on the next flight and issue the credit I am owed.",
                }
            ]
        },
    )
    pending = result.get("__interrupt__")
    if pending:
        for request in pending[0]["value"]["action_requests"]:
            print("Waiting for approval:", request["name"], request["args"])
    else:
        print(reply(result))

The interrupt carries the tool call waiting for approval. Resume the same thread with a
`Command`: approve it, or send `{"type": "reject", "message": "..."}`.

In [ ]:
# needs: deployment
if client and pending:
    resumed = await client.runs.wait(
        thread["thread_id"],
        "rebooking",
        command={"resume": {"decisions": [{"type": "approve"}]}},
    )
    print(reply(resumed))
elif client:
    print("The agent did not ask for approval this time. Run the previous cell again.")

## Under the hood

<img src="../../assets/slides/langgraph-human-in-the-loop.jpg" width="720">

- **The client is a plain HTTP API.** `threads`, `runs`, and `assistants` are Agent Server
  endpoints. Anything that speaks HTTP can call the agent. The SDK is a convenience.
- **An interrupt is a pause, not a failure.** The run returns `__interrupt__`, the thread's state
  is saved, and `Command(resume=...)` continues from the exact checkpoint. The approval can come
  minutes or days later, from a different process.
- **`assistant_id` can be the graph id.** `"rebooking"` is the key in `langgraph.json`.
- **Every run is traced.** No tracing code in the client. The server writes the trace to the
  deployment's project.

Docs: [Human-in-the-loop](https://docs.langchain.com/oss/python/langgraph/interrupts).

## Recap

- A deployment is the same agent behind an HTTP API.
- Threads hold conversations. Runs execute on them.
- An interrupt pauses a run until `Command(resume=...)` answers it.

**Next:** [3.2 Deploy](02_deploy.ipynb)